In [1]:
# Load the essential libraries

%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from platform import python_version
import tensorflow as tf
from pyod.models.auto_encoder import AutoEncoder

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

print(f'TensorFlow version: {tf.__version__}')
print(f'Python version: {python_version()}')

2025-11-22 11:34:38.200457: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-22 11:34:38.521076: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/peera/ros2_ws/install/v4l2_camera/lib:/home/peera/ws_moveit/install/pilz_industrial_motion_planner_testutils/lib:/home/peera/ws_moveit/install/pilz_industrial_motion_planner/lib:/home/peera/ws_moveit/install/moveit_visual_tools/lib:/home/peera/ws_moveit/install/moveit_task_constructor_visualization/lib:/home/peera/ws_moveit/install/moveit_task_constructor_demo/lib:/home/peera/ws_moveit/in

TensorFlow version: 2.11.0
Python version: 3.10.12


## Prepare Data

In [2]:
# Load and preview the dataset

df = pd.read_csv('creditcard.csv')
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
# Assign model features and the target label column to variables

model_features = df.columns.drop('Class')
X = df[model_features]
y = df['Class']

In [4]:
# View the target label distribution

y.value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

In [5]:
# View the number of rows and columns of model features

X.shape

(284807, 30)

## Train a Model

In [6]:
# Set the contamination and epochs. Contamination ranges between 0 and 0.5 with 0.1 as default value.

contamination = 0.1 
epochs = 30

In [ ]:
# Set the number of neurons per hidden layers

hn = [20, 10, 3, 10, 20]

# Instantiate AutoEncoder and start training

clf = AutoEncoder(epoch_num=epochs, contamination=contamination, hidden_neuron_list=hn)
clf.fit(X)

Training:   7%|████████▋                                                                                                                         | 2/30 [01:10<16:29, 35.34s/it]

## Identify Outliers

In [ ]:
# Obtain prediction on outliers

outliers = clf.predict(X)

In [ ]:
# Filter outliers

anomaly = np.where(outliers==1)
anomaly

In [ ]:
# Predict a test instance

sample = X.iloc[[4920]]

clf.predict(sample, return_confidence=False)

In [ ]:
X.iloc[[4920]]

In [ ]:
y.iloc[[4920]]

In [ ]:
# Ground truth fraudulent transactions

df.loc[df['Class'] == 1]

In [ ]:
# Show model's confidence if trained using perturbed data

clf.predict_confidence(sample)

In [ ]:
# Generate binary labels of the training data. 0 means inliers and 1 means outliers

y_pred = clf.labels_  

# Calculate outlier scores of the training data. Higher scores means higher severity of abnormalities

y_scores = clf.decision_scores_ 

In [ ]:
y_pred[:5]

In [ ]:
y_scores[:5]

## Evaluate Model

In [ ]:
# Visualize auto-calculated anomaly scores. Red horizontal line represents threshold.

plt.rcParams["figure.figsize"] = (15,8)
plt.plot(y_scores);
plt.axhline(y=clf.threshold_, c='r', ls='dotted', label='threshold');
plt.xlabel('Instances')
plt.ylabel('Anomaly Scores')
plt.title('Anomaly Scores with Auto-Calculated Threshold');
plt.show()

In [ ]:
# Modify the threshold and view the new distribution. Red horizontal line represents the manually changed threshold.

threshold = 50
plt.rcParams["figure.figsize"] = (15,8)
plt.plot(y_scores, color="green");
plt.axhline(y=threshold, c='r', ls='dotted', label='threshold');
plt.xlabel('Instances')
plt.ylabel('Anomaly Scores')
plt.title('Anomaly Scores with Modified Threshold');
plt.show()

In [ ]:
# Visualize a scatter plot

sns.scatterplot(x="Time", y="Amount", hue=y_scores, data=df, palette="RdBu_r", size=y_scores);
plt.xlabel('Time (seconds elapsed from first transaction)')
plt.ylabel('Amount')
plt.legend(title='Anomaly Scores')
plt.show()